In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/processed/unsw_nb15_processed.csv")

In [ ]:
df['is_attack'] = df['label']

In [ ]:
df[['srcip;, 'dstip']].head()

In [ ]:
threat_intel_db = {
    "175.45.176.2": {"reputation": "malicious", "confidence": 90},
    "185.220.101.1": {"reputation": "malicious", "confidence": 85},
    "91.240.118.172": {"reputation": "malicious", "confidence": 60},
    "10.0.0.5": {"reputation": "internal", "confidence":0}
}

In [ ]:
def lookup_ip_reputation(ip):
    if ip in threat_intel_db:
        return threat_intel_db[ip]['reputation'], threat_intel_db[ip]['confidence']
    else:
        return "unknown", 0

In [ ]:
df[['src_reputation', 'src_confidence']] = df['dstip'].apply(
    lambda x: pd.Series(lookup_ip_reputation(x))
)

In [ ]:
df[['srcip', 'src_reputation', 'src_confidence', 'label']].head(10)

In [ ]:
def calculate_risk(row):
    risk = 0

    if row['label'] == 1:
        risk += 50

    if row['src_reputation'] == 'malicious':
        risk += row['src_confidence']
    elif row['src_reputation'] == 'suspicious':
        risk += 30

    return min(risk, 100)

In [ ]:
df['risk_score'] = df.apply(calculate_risk, axis=1)

In [ ]:
def classify_severity(score):
    if score >= 80:
        return "Critical"
    elif score >= 50:
        return "High"
    elif score >= 20:
        return "Medium"
    else:
        return "Low"

In [ ]:
df[['alert_severity'] = df['risk_score'].apply(classify_severity)

In [ ]:
df[['srcip', 'label', 'src_reputation', 'risk_score', 'alert_severity']].sample(10)

In [ ]:
df.to_csv("../data/processed/unsw_nb15_enriched.csv", index=False)